# 04 — Explore notifications and use the saved model
The app has three tabs:
1. **Explore notifications:** year/date range, product type, category, notifying country, origin, classification, risk label and literal text filters; charts, records and CSV export.
2. **Predict recorded label:** dynamically generated inputs matching the selected saved model, including original subject and hazards. Multiple origins are supported.
3. **Model & evaluation:** candidate, test dates, metrics, confusion matrix and scope.

The filters affect descriptive data only. They do not secretly retrain or subset the classifier. Generated hazard fields are not required in the default app.

In [1]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
from IPython.display import display
ROOT = Path.cwd()
if not (ROOT / "rasff_core.py").exists():
    raise FileNotFoundError("Open this notebook from the extracted RASFF_Rebuild folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from rasff_core import *


## 1. Check artifacts and the input contract

In [2]:
needed = [DATA / "prepared.csv", ARTIFACTS / "final_bundle.joblib", ARTIFACTS / "input_options.json", ARTIFACTS / "test_report.json"]
for p in needed:
    print(p.name, "OK" if p.exists() else "MISSING")
assert all(p.exists() for p in needed), "Execute 01–03 first."
bundle = joblib.load(ARTIFACTS / "final_bundle.joblib")
display(pd.Series(bundle["columns"],name="App inputs"))

prepared.csv OK
final_bundle.joblib OK
input_options.json OK
test_report.json OK


0              subject
1               origin
2             category
3                 type
4    notifying_country
5              hazards
Name: App inputs, dtype: object

## 2. Launch from a terminal
Run the command below from the same extracted folder:
```bash
python -m streamlit run app.py
```
Open the local URL printed by Streamlit, usually http://localhost:8501. Use Ctrl+C in the terminal to stop it. Do not launch a blocking server directly inside a notebook cell.

## 3. Verify the app uses the same function
`app.py` calls `predict_bundle`. Notebook and app therefore share the saved pipeline, calibrator and required columns. The following represents an existing record for reproducibility, not a new prospective prediction.

In [3]:
df = load_prepared()
sample = df.iloc[[0]]
print("Shared prediction function:",float(predict_bundle(bundle,sample)[0]))
print("App target: recorded serious, NOT recall probability")

Shared prediction function: 0.9013031812634226
App target: recorded serious, NOT recall probability


## 4. Optional automated UI smoke test
This checks rendering, date/year filters and a real prediction through the form. It does not open a browser or publish the app. For a manual end-to-end check, select an existing notification and compare its values in notebook and UI.

In [4]:
RUN_UI_SMOKE_TEST = False
if RUN_UI_SMOKE_TEST:
    from streamlit.testing.v1 import AppTest
    at = AppTest.from_file(str(ROOT / "app.py"),default_timeout=60).run()
    assert not at.exception, at.exception
    print("Streamlit rendered without errors.")

## Blog notes
Demonstrate a year/date/category filter and explain what counts mean. Then demonstrate one prediction with the actual selected input fields. Explain why the three-input historical mockup was replaced by the exact model input contract. No official risk judgement or validated pre-notification prediction is implied.